# Neurotoxicity Profiler — Computational Tool
## Full Pipeline: SMILES/InChI → ToxCast/Tox21 → Risk Score + Hazard Flag

**Domain:** Computational Toxicology / New Approach Methods (NAM)  
**Data sources:** EPA ToxCast · Tox21 · CompTox · PubChem  
**Methods:** Molecular fingerprints · QSAR · Random Forest/XGBoost · AOP mapping  

---

### Architecture
```
Input: SMILES / InChI / DTXSID
    ↓
[Module 1]  Structure ingestion & validation (RDKit)
    ↓
[Module 2]  Molecular feature engineering (fingerprints + physicochemical)
    ↓
[Module 3]  ToxCast/Tox21 assay data retrieval (EPA CompTox API)
    ↓
[Module 4]  Neurotoxicity-specific assay panel filtering (AOP-anchored)
    ↓
[Module 5]  ML risk scoring (RF + XGBoost ensemble)
    ↓
[Module 6]  Hazard flagging + confidence estimation
    ↓
[Module 7]  Mechanistic pathway annotation (AOP/GO)
    ↓
Output: NeurotoxicityProfile (risk score, hazard flag, assay hits, pathways)
```

### Install
```bash
pip install rdkit-pypi pandas numpy scikit-learn xgboost matplotlib seaborn requests
conda install -c conda-forge rdkit  # preferred for RDKit
```

---
## Module 1: Chemical Structure Ingestion & Validation

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, rdMolDescriptors, AllChem, inchi
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import pandas as pd
import numpy as np
import requests
import json
import re
import time
import warnings
warnings.filterwarnings('ignore')
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field, asdict


@dataclass
class ChemicalRecord:
    """Validated chemical record ready for profiling."""
    input_id:      str                    # original input identifier
    smiles:        Optional[str] = None
    inchi:         Optional[str] = None
    inchikey:      Optional[str] = None
    dtxsid:        Optional[str] = None   # EPA DSSTox identifier
    name:          Optional[str] = None
    mol:           object        = field(default=None, repr=False)
    valid:         bool          = False
    error:         Optional[str] = None


def parse_smiles(smiles: str, input_id: str = '') -> ChemicalRecord:
    """Parse and sanitize a SMILES string."""
    rec = ChemicalRecord(input_id=input_id or smiles, smiles=smiles)
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            rec.error = 'Invalid SMILES — could not parse'
            return rec
        Chem.SanitizeMol(mol)
        rec.mol       = mol
        rec.smiles    = Chem.MolToSmiles(mol)        # canonical SMILES
        rec.inchi     = inchi.MolToInchi(mol)
        rec.inchikey  = inchi.InchiToInchiKey(rec.inchi) if rec.inchi else None
        rec.valid     = True
    except Exception as e:
        rec.error = str(e)
    return rec


def parse_inchi(inchi_str: str, input_id: str = '') -> ChemicalRecord:
    """Parse an InChI string and convert to RDKit mol."""
    rec = ChemicalRecord(input_id=input_id or inchi_str, inchi=inchi_str)
    try:
        mol = inchi.MolFromInchi(inchi_str)
        if mol is None:
            rec.error = 'Invalid InChI'
            return rec
        rec.mol      = mol
        rec.smiles   = Chem.MolToSmiles(mol)
        rec.inchikey = inchi.InchiToInchiKey(inchi_str)
        rec.valid    = True
    except Exception as e:
        rec.error = str(e)
    return rec


def parse_batch(records: List[Dict]) -> List[ChemicalRecord]:
    """
    Parse a batch of chemicals from a list of dicts.
    Each dict must have 'id' and either 'smiles' or 'inchi'.
    """
    parsed = []
    for r in records:
        cid = r.get('id', str(len(parsed)))
        if 'smiles' in r:
            parsed.append(parse_smiles(r['smiles'], cid))
        elif 'inchi' in r:
            parsed.append(parse_inchi(r['inchi'], cid))
        else:
            parsed.append(ChemicalRecord(input_id=cid,
                                          error='No smiles or inchi provided'))
    return parsed


# ── Known neurotoxic reference set ─────────────────────────────────────────────
# Includes established neurotoxicants and structurally diverse controls
# Sources: IRIS, NTP, ECHA CMR, EFSA EFAS
TEST_CHEMICALS = [
    {'id':'PFOA',        'name':'Perfluorooctanoic acid',
     'smiles':'OC(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F',
     'neuro_label':1},
    {'id':'Chlorpyrifos','name':'Chlorpyrifos (organophosphate)',
     'smiles':'CCOP(=S)(OCC)Oc1nc(Cl)c(Cl)cc1Cl',
     'neuro_label':1},
    {'id':'MeHg_proxy', 'name':'Methylmercury proxy (CH3HgCl)',
     'smiles':'C[Hg]Cl',
     'neuro_label':1},
    {'id':'Lead_acetate','name':'Lead(II) acetate',
     'smiles':'CC(=O)O[Pb]OC(C)=O',
     'neuro_label':1},
    {'id':'BPA',         'name':'Bisphenol A',
     'smiles':'CC(C)(c1ccc(O)cc1)c1ccc(O)cc1',
     'neuro_label':1},
    {'id':'Deltamethrin','name':'Deltamethrin (pyrethroid)',
     'smiles':'CC1(C)[C@@H](C=C(Br)Br)[C@H]1C(=O)O[C@@H](C#N)c1cccc(Oc2ccccc2)c1',
     'neuro_label':1},
    {'id':'Rotenone',    'name':'Rotenone (Complex I inhibitor)',
     'smiles':'O=C1OC2=CC(=CC=C2[C@@H]1CC1=CC2=C(C=C1)OCO2)OC',
     'neuro_label':1},
    {'id':'MPP_plus',    'name':'MPP+ (dopaminergic neurotoxin)',
     'smiles':'C[n+]1ccc(cc1)C=O',
     'neuro_label':1},
    {'id':'Sucrose',     'name':'Sucrose (negative control)',
     'smiles':'OC[C@H]1O[C@@](CO)(O[C@H]2O[C@@H](CO)[C@@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@@H]1O',
     'neuro_label':0},
    {'id':'Aspirin',     'name':'Aspirin (negative control)',
     'smiles':'CC(=O)Oc1ccccc1C(=O)O',
     'neuro_label':0},
    {'id':'Caffeine',    'name':'Caffeine (low neuro concern)',
     'smiles':'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
     'neuro_label':0},
    {'id':'DEET',        'name':'DEET (insect repellent)',
     'smiles':'CCN(CC)C(=O)c1cccc(C)c1',
     'neuro_label':0},
]

chem_records = parse_batch(TEST_CHEMICALS)
labels       = {r['id']: r['neuro_label'] for r in TEST_CHEMICALS}
names        = {r['id']: r['name']        for r in TEST_CHEMICALS}

valid   = [r for r in chem_records if r.valid]
invalid = [r for r in chem_records if not r.valid]

print(f'Parsed {len(chem_records)} chemicals')
print(f'  Valid:   {len(valid)}')
print(f'  Invalid: {len(invalid)}')
if invalid:
    for r in invalid:
        print(f'    {r.input_id}: {r.error}')
print()
for r in valid:
    print(f'  {r.input_id:15s}  InChIKey: {r.inchikey}  Label: {labels.get(r.input_id,"?")}')

---
## Module 2: Molecular Feature Engineering

In [ ]:
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs


# ── Feature set design ────────────────────────────────────────────────────────
# 1. Morgan fingerprints (ECFP4-like)  — 2048 bits, captures local structure
# 2. MACCS keys                        — 167 bits, interpretable substructure keys
# 3. RDKit fingerprints                — topological, path-based
# 4. Physicochemical descriptors       — 12 curated for neurotox relevance
#
# Neurotox-relevant physicochemical properties:
#   BBB penetration: logP, MW, TPSA, HBD (blood-brain barrier crossing)
#   Reactivity:      num rings, aromaticity, halogen count
#   Lipophilicity:   MolLogP, MolMR

def compute_morgan_fp(mol, radius: int = 2, nbits: int = 2048) -> np.ndarray:
    """ECFP4-equivalent Morgan fingerprint."""
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
    return np.array(fp)


def compute_maccs_fp(mol) -> np.ndarray:
    """MACCS keys — 167 structural keys, interpretable."""
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)


def compute_rdkit_fp(mol, nbits: int = 2048) -> np.ndarray:
    """RDKit topological fingerprint."""
    fp = Chem.RDKFingerprint(mol, fpSize=nbits)
    return np.array(fp)


NEUROTOX_DESCRIPTORS = [
    ('MW',          Descriptors.MolWt),
    ('LogP',        Descriptors.MolLogP),
    ('TPSA',        lambda m: CalcTPSA(m)),
    ('HBD',         rdMolDescriptors.CalcNumHBD),
    ('HBA',         rdMolDescriptors.CalcNumHBA),
    ('RotBonds',    rdMolDescriptors.CalcNumRotatableBonds),
    ('AromaticRings',rdMolDescriptors.CalcNumAromaticRings),
    ('Rings',       rdMolDescriptors.CalcNumRings),
    ('HeavyAtoms',  Descriptors.HeavyAtomCount),
    ('MolMR',       Descriptors.MolMR),
    ('FractionCSP3',rdMolDescriptors.CalcFractionCSP3),
    ('Halogens',    lambda m: sum(1 for a in m.GetAtoms()
                                  if a.GetAtomicNum() in (9,17,35,53))),
    ('HeavyMetals', lambda m: sum(1 for a in m.GetAtoms()
                                  if a.GetAtomicNum() in (80,82,33,48,24,28))),
]


def compute_physicochemical(mol) -> Dict[str, float]:
    """Compute curated physicochemical descriptors for neurotoxicity."""
    desc = {}
    for name, func in NEUROTOX_DESCRIPTORS:
        try:
            desc[name] = float(func(mol))
        except Exception:
            desc[name] = np.nan
    return desc


def featurize(record: ChemicalRecord,
              fp_type: str = 'morgan',
              include_physchem: bool = True) -> Optional[np.ndarray]:
    """
    Generate feature vector for a validated chemical record.

    fp_type options: 'morgan' | 'maccs' | 'rdkit' | 'combined'
    combined = Morgan(1024) + MACCS(167) + physchem(13)
    """
    if not record.valid or record.mol is None:
        return None
    mol = record.mol

    if fp_type == 'morgan':
        fp = compute_morgan_fp(mol, nbits=1024)
    elif fp_type == 'maccs':
        fp = compute_maccs_fp(mol)
    elif fp_type == 'rdkit':
        fp = compute_rdkit_fp(mol, nbits=1024)
    elif fp_type == 'combined':
        fp = np.concatenate([
            compute_morgan_fp(mol, nbits=1024),
            compute_maccs_fp(mol)
        ])
    else:
        raise ValueError(f'Unknown fp_type: {fp_type}')

    if include_physchem:
        phys  = compute_physicochemical(mol)
        phys_vec = np.array([phys.get(n, 0.0) for n, _ in NEUROTOX_DESCRIPTORS])
        # Normalise physicochemical features
        norms = np.array([500., 5., 140., 5., 10., 15., 4., 6., 50., 80., 1., 9., 3.])
        phys_vec = np.clip(phys_vec / norms, 0, 3)
        fp = np.concatenate([fp, phys_vec])

    return fp.astype(np.float32)


# ── Compute features for all valid records ──────────────────────────────────
feature_matrix = []
feature_ids    = []
feature_labels = []
physchem_rows  = []

for rec in valid:
    vec = featurize(rec, fp_type='combined', include_physchem=True)
    if vec is not None:
        feature_matrix.append(vec)
        feature_ids.append(rec.input_id)
        feature_labels.append(labels.get(rec.input_id, -1))
        physchem_rows.append(compute_physicochemical(rec.mol))

X = np.array(feature_matrix)
y = np.array(feature_labels)

physchem_df = pd.DataFrame(physchem_rows, index=feature_ids)

print(f'Feature matrix: {X.shape}  ({X.shape[1]} features per chemical)')
print(f'  Morgan(1024) + MACCS(167) + physchem({len(NEUROTOX_DESCRIPTORS)})')
print()
print('Physicochemical descriptors:')
print(physchem_df.round(2).to_string())

---
## Module 3: ToxCast / Tox21 Assay Data Integration

In [ ]:
# ── EPA CompTox Chemicals Dashboard API ────────────────────────────────────────
# Endpoint: https://comptox.epa.gov/dashboard-api/
# Documentation: https://api-ccte.epa.gov/docs/

COMPTOX_BASE = 'https://api-ccte.epa.gov'

def get_toxcast_data_by_inchikey(inchikey: str,
                                  timeout: int = 10) -> Optional[Dict]:
    """
    Query EPA CompTox API for ToxCast assay activity data.
    Returns dict of assay_name -> activity_flag (1=active, 0=inactive).

    Requires free API key from: https://api-ccte.epa.gov/
    Set: API_KEY = os.getenv('COMPTOX_API_KEY', '')
    """
    # In production: add API key header
    # headers = {'x-api-key': API_KEY}
    url = f'{COMPTOX_BASE}/bioactivity/data/search/by-inchikey/{inchikey}'
    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code == 200:
            return resp.json()
    except Exception:
        pass
    return None


def get_dtxsid_by_inchikey(inchikey: str) -> Optional[str]:
    """Look up EPA DSSTox DTXSID from InChIKey."""
    url = f'{COMPTOX_BASE}/chemical/detail/search/by-inchikey/{inchikey}'
    try:
        resp = requests.get(url, timeout=8)
        if resp.status_code == 200:
            data = resp.json()
            if isinstance(data, list) and data:
                return data[0].get('dtxsid')
    except Exception:
        pass
    return None


# ── Neurotoxicity-relevant ToxCast assay panel ─────────────────────────────────
# Selected from ToxCast Phase I/II covering key neurotox mechanisms:
# Ion channel disruption, cholinesterase inhibition, oxidative stress,
# dopaminergic signalling, thyroid disruption (T4/T3 — indirect neuro effect),
# developmental endpoints

NEURO_ASSAY_PANEL = {
    # Acetylcholinesterase / cholinergic system
    'NVS_ENZ_hAChE':             {'mechanism': 'AChE_inhibition',     'weight': 3.0},
    'Tox21_AChE_Inhibition':     {'mechanism': 'AChE_inhibition',     'weight': 3.0},
    'NVS_ENZ_rAChE':             {'mechanism': 'AChE_inhibition',     'weight': 2.5},
    # Voltage-gated ion channels
    'NVS_IC_hKhERG':             {'mechanism': 'hERG_channel',        'weight': 2.0},
    'NVS_IC_rNaVt':              {'mechanism': 'Na_channel',          'weight': 2.5},
    'Tox21_hERG_BLA_Agonist':    {'mechanism': 'hERG_channel',        'weight': 2.0},
    # Dopaminergic system
    'NVS_GPCR_hDAT':             {'mechanism': 'dopamine_transport',  'weight': 2.5},
    'NVS_GPCR_hD2s':             {'mechanism': 'D2_receptor',         'weight': 2.0},
    'CEETOX_HTRF_DAT_Inh':       {'mechanism': 'dopamine_transport',  'weight': 2.5},
    # Oxidative stress / mitochondria
    'Tox21_ARE_BLA_Agonist':     {'mechanism': 'oxidative_stress',    'weight': 2.0},
    'NVS_ADME_hCYP2C19':         {'mechanism': 'CYP_metabolism',      'weight': 1.0},
    'Tox21_MitoMembPot':         {'mechanism': 'mitochondrial',       'weight': 2.5},
    # Thyroid axis (indirect neurodevelopment)
    'Tox21_TR_BLA_Agonist':      {'mechanism': 'thyroid_receptor',    'weight': 1.5},
    'NVS_NR_hTRa_Antagonist':    {'mechanism': 'thyroid_receptor',    'weight': 1.5},
    # GABA receptor
    'NVS_LG_rGABARa1':           {'mechanism': 'GABA_receptor',       'weight': 2.0},
    # Developmental / neurotrophic
    'TOX21_NFKB_BLA_Agonist':    {'mechanism': 'neuroinflammation',   'weight': 1.5},
    'Tox21_p53_BLA_Agonist':     {'mechanism': 'genotoxicity',        'weight': 1.0},
}

# Adverse Outcome Pathway (AOP) mechanistic groupings
# AOPs from AOP-Wiki relevant to neurotoxicity
AOP_MAPPING = {
    'AChE_inhibition':    {'aop_id': 'AOP-18', 'title': 'AChE inhibition → acute cholinergic syndrome',
                           'mie': 'AChE inhibition', 'ao': 'Neurological dysfunction'},
    'dopamine_transport': {'aop_id': 'AOP-3',  'title': 'DAT inhibition → Parkinsonian motor deficits',
                           'mie': 'DAT/D2 inhibition', 'ao': 'Dopaminergic neurotoxicity'},
    'mitochondrial':      {'aop_id': 'AOP-53', 'title': 'Complex I inhibition → neurodegeneration',
                           'mie': 'Mitochondrial Complex I inhibition', 'ao': 'Neuronal cell death'},
    'oxidative_stress':   {'aop_id': 'AOP-98', 'title': 'Oxidative stress → neuroinflammation',
                           'mie': 'ROS generation / Nrf2 activation', 'ao': 'Neuroinflammation'},
    'Na_channel':         {'aop_id': 'AOP-14', 'title': 'Nav activation → seizure/epilepsy',
                           'mie': 'Voltage-gated sodium channel activation', 'ao': 'Seizure'},
    'thyroid_receptor':   {'aop_id': 'AOP-42', 'title': 'Thyroid hormone disruption → neurodevelopment',
                           'mie': 'TR agonism/antagonism', 'ao': 'Neurodevelopmental impairment'},
    'GABA_receptor':      {'aop_id': 'AOP-57', 'title': 'GABAR inhibition → seizure',
                           'mie': 'GABA-A receptor antagonism', 'ao': 'Seizure / hyperexcitability'},
    'neuroinflammation':  {'aop_id': 'AOP-12', 'title': 'NF-kB activation → neuroinflammation',
                           'mie': 'NF-kB signalling', 'ao': 'Neuroinflammation'},
    'hERG_channel':       {'aop_id': None, 'title': 'hERG inhibition → cardiac arrhythmia (indirect CNS)',
                           'mie': 'hERG potassium channel block', 'ao': 'Cardiac / indirect CNS effect'},
    'D2_receptor':        {'aop_id': 'AOP-3',  'title': 'D2R dysregulation → dopaminergic toxicity',
                           'mie': 'D2 receptor antagonism', 'ao': 'Dopaminergic neurotoxicity'},
}


# ── Simulated ToxCast activity data ───────────────────────────────────────────
# Realistic AC50-based activity flags (1=active, 0=inactive)
# In production: replace with live CompTox API calls

SIMULATED_TOXCAST = {
    'PFOA':         {'NVS_ENZ_hAChE':0,'NVS_GPCR_hDAT':0,'Tox21_MitoMembPot':1,
                     'Tox21_TR_BLA_Agonist':1,'NVS_NR_hTRa_Antagonist':1,
                     'TOX21_NFKB_BLA_Agonist':1,'NVS_LG_rGABARa1':0,
                     'NVS_IC_rNaVt':0,'CEETOX_HTRF_DAT_Inh':0},
    'Chlorpyrifos': {'NVS_ENZ_hAChE':1,'Tox21_AChE_Inhibition':1,'NVS_ENZ_rAChE':1,
                     'NVS_GPCR_hDAT':0,'Tox21_MitoMembPot':1,
                     'NVS_IC_rNaVt':1,'NVS_LG_rGABARa1':0,'TOX21_NFKB_BLA_Agonist':0},
    'MeHg_proxy':   {'NVS_ENZ_hAChE':0,'Tox21_MitoMembPot':1,'Tox21_ARE_BLA_Agonist':1,
                     'TOX21_NFKB_BLA_Agonist':1,'NVS_GPCR_hDAT':1,
                     'CEETOX_HTRF_DAT_Inh':1,'NVS_LG_rGABARa1':0,'NVS_IC_rNaVt':0},
    'Lead_acetate': {'NVS_ENZ_hAChE':0,'Tox21_MitoMembPot':1,'Tox21_ARE_BLA_Agonist':1,
                     'NVS_GPCR_hDAT':1,'TOX21_NFKB_BLA_Agonist':1,'NVS_LG_rGABARa1':1,
                     'NVS_NR_hTRa_Antagonist':1,'NVS_IC_rNaVt':0},
    'BPA':          {'Tox21_TR_BLA_Agonist':1,'NVS_NR_hTRa_Antagonist':0,
                     'TOX21_NFKB_BLA_Agonist':1,'Tox21_ARE_BLA_Agonist':1,
                     'NVS_ENZ_hAChE':0,'Tox21_MitoMembPot':0,'NVS_GPCR_hDAT':0},
    'Deltamethrin': {'NVS_IC_rNaVt':1,'Tox21_AChE_Inhibition':0,'NVS_GPCR_hDAT':0,
                     'Tox21_MitoMembPot':0,'NVS_LG_rGABARa1':1,'NVS_ENZ_hAChE':0,
                     'TOX21_NFKB_BLA_Agonist':0},
    'Rotenone':     {'Tox21_MitoMembPot':1,'NVS_GPCR_hDAT':1,'Tox21_ARE_BLA_Agonist':1,
                     'CEETOX_HTRF_DAT_Inh':1,'TOX21_NFKB_BLA_Agonist':1,
                     'NVS_ENZ_hAChE':0,'NVS_IC_rNaVt':0,'NVS_LG_rGABARa1':0},
    'MPP_plus':     {'NVS_GPCR_hDAT':1,'CEETOX_HTRF_DAT_Inh':1,'NVS_GPCR_hD2s':1,
                     'Tox21_MitoMembPot':1,'Tox21_ARE_BLA_Agonist':1,
                     'NVS_ENZ_hAChE':0,'NVS_LG_rGABARa1':0,'NVS_IC_rNaVt':0},
    'Sucrose':      {'NVS_ENZ_hAChE':0,'NVS_GPCR_hDAT':0,'Tox21_MitoMembPot':0,
                     'Tox21_ARE_BLA_Agonist':0,'NVS_LG_rGABARa1':0,'NVS_IC_rNaVt':0,
                     'TOX21_NFKB_BLA_Agonist':0,'Tox21_TR_BLA_Agonist':0},
    'Aspirin':      {'NVS_ENZ_hAChE':0,'Tox21_MitoMembPot':0,'Tox21_ARE_BLA_Agonist':1,
                     'NVS_GPCR_hDAT':0,'NVS_LG_rGABARa1':0,'TOX21_NFKB_BLA_Agonist':1,
                     'NVS_IC_rNaVt':0,'Tox21_TR_BLA_Agonist':0},
    'Caffeine':     {'NVS_ENZ_hAChE':0,'NVS_GPCR_hDAT':0,'Tox21_MitoMembPot':0,
                     'NVS_LG_rGABARa1':0,'NVS_IC_rNaVt':0,'TOX21_NFKB_BLA_Agonist':0,
                     'Tox21_TR_BLA_Agonist':0,'Tox21_ARE_BLA_Agonist':0},
    'DEET':         {'NVS_ENZ_hAChE':1,'Tox21_AChE_Inhibition':0,'NVS_GPCR_hDAT':0,
                     'Tox21_MitoMembPot':0,'NVS_LG_rGABARa1':0,'NVS_IC_rNaVt':0,
                     'TOX21_NFKB_BLA_Agonist':0,'Tox21_TR_BLA_Agonist':0},
}

# Build assay feature matrix (aligned to NEURO_ASSAY_PANEL)
assay_names = sorted(NEURO_ASSAY_PANEL.keys())
assay_matrix = []
assay_ids    = []

for cid in feature_ids:
    row = SIMULATED_TOXCAST.get(cid, {})
    assay_matrix.append([float(row.get(a, 0)) for a in assay_names])
    assay_ids.append(cid)

assay_df = pd.DataFrame(assay_matrix, index=assay_ids, columns=assay_names)

print('ToxCast assay activity matrix (1=active, 0=inactive):')
print(assay_df.to_string())
print(f'\nTotal assay hits per chemical:')
print((assay_df.sum(axis=1)).sort_values(ascending=False))

---
## Module 4: Weighted Neurotoxicity Assay Score

In [ ]:
def compute_assay_score(assay_hits: Dict[str, int],
                         panel: Dict[str, Dict]) -> Dict:
    """
    Compute a weighted neurotoxicity concern score from assay activity.

    Score = sum(weight_i * hit_i) / sum(all_weights) * 100
    Range: 0 (no activity) to 100 (all weighted assays active)
    """
    total_weight = sum(v['weight'] for v in panel.values())
    hit_weight   = sum(
        panel[a]['weight'] for a in assay_hits
        if assay_hits[a] == 1 and a in panel
    )
    assay_score  = (hit_weight / total_weight) * 100

    # Mechanisms triggered
    mechanisms = list(set(
        panel[a]['mechanism'] for a in assay_hits
        if assay_hits[a] == 1 and a in panel
    ))

    # AOP annotations
    aops_triggered = []
    for mech in mechanisms:
        if mech in AOP_MAPPING:
            aops_triggered.append(AOP_MAPPING[mech])

    return {
        'assay_score':      round(assay_score, 2),
        'n_hits':           sum(1 for v in assay_hits.values() if v == 1),
        'n_assays_tested':  len(assay_hits),
        'mechanisms':       mechanisms,
        'aops_triggered':   aops_triggered,
        'hit_assays':       [a for a, v in assay_hits.items() if v == 1],
    }


assay_scores = {}
for cid, row in assay_df.iterrows():
    assay_scores[cid] = compute_assay_score(row.to_dict(), NEURO_ASSAY_PANEL)

print('Weighted assay scores (0-100) and triggered mechanisms:')
print('='*70)
for cid, sc in sorted(assay_scores.items(), key=lambda x: -x[1]['assay_score']):
    print(f'  {cid:15s}  score={sc["assay_score"]:5.1f}  hits={sc["n_hits"]}  '
          f'mechanisms: {", ".join(sc["mechanisms"][:3]) or "none"}')

---
## Module 5: ML Risk Scoring — RF + XGBoost Ensemble

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, roc_auc_score,
                               precision_recall_curve, average_precision_score)
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns

# NOTE: 12 chemicals is far too few for production ML.
# This notebook demonstrates the ARCHITECTURE.
# Production requires 500+ chemicals from:
#   - ToxRefDB (in vivo), Tox21 (>8000 chemicals), ECOTOX, CCRIS

# ── Combined feature matrix: molecular fingerprints + assay data ──────────────
# In production: train on ToxCast Phase I/II full dataset (~1800 chemicals)

X_combined = np.hstack([
    X,                  # Morgan + MACCS + physicochemical
    assay_df.values     # ToxCast assay activity flags
])
print(f'Combined feature matrix: {X_combined.shape}')
print(f'  Molecular fingerprints + physchem: {X.shape[1]}')
print(f'  ToxCast assay flags:               {assay_df.shape[1]}')
print(f'  Labels (1=neurotoxic, 0=control):  {dict(zip(*np.unique(y, return_counts=True)))}')


# ── Model definitions ─────────────────────────────────────────────────────────
RF_MODEL = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        class_weight='balanced',   # important: neurotox datasets are imbalanced
        random_state=42,
        n_jobs=-1
    ))
])

try:
    import xgboost as xgb
    XGB_MODEL = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,   # adjust for class imbalance
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    )
    has_xgb = True
    print('XGBoost available.')
except ImportError:
    has_xgb = False
    print('XGBoost not installed. Using GradientBoosting fallback.')
    XGB_MODEL = GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
    )


# ── Train both models on full small dataset (demo) ────────────────────────────
RF_MODEL.fit(X_combined, y)
XGB_MODEL.fit(X_combined, y)

rf_proba  = RF_MODEL.predict_proba(X_combined)[:, 1]
xgb_proba = XGB_MODEL.predict_proba(X_combined)[:, 1]

# Ensemble: average probabilities
ensemble_proba = (rf_proba + xgb_proba) / 2

print('\nPer-chemical probability (RF | XGB | Ensemble | True Label):')
print('-'*65)
for i, cid in enumerate(feature_ids):
    print(f'  {cid:15s}  RF={rf_proba[i]:.3f}  XGB={xgb_proba[i]:.3f}  '
          f'Ens={ensemble_proba[i]:.3f}  Label={y[i]}')

---
## Module 6: Risk Score Computation & Hazard Flagging

In [ ]:
# ── NeurotoxicityProfile dataclass ────────────────────────────────────────────
@dataclass
class NeurotoxicityProfile:
    """
    Complete neurotoxicity assessment output for a single chemical.
    Output of the full profiler pipeline.
    """
    chemical_id:      str
    name:             Optional[str]
    smiles:           Optional[str]
    inchikey:         Optional[str]
    # Risk score (0-100)
    ml_score:         float           # ML ensemble probability * 100
    assay_score:      float           # Weighted assay activity score (0-100)
    composite_score:  float           # Weighted combination of both
    # Hazard classification
    hazard_flag:      str             # HIGH / MEDIUM / LOW / NEGATIVE
    confidence:       str             # HIGH / MODERATE / LOW
    # Evidence
    n_assay_hits:     int
    mechanisms:       List[str]       = field(default_factory=list)
    aops_triggered:   List[Dict]      = field(default_factory=list)
    key_assay_hits:   List[str]       = field(default_factory=list)
    # Physicochemical flags
    bbb_concern:      bool = False    # LogP>2 AND MW<500 AND TPSA<90
    heavy_metal:      bool = False    # contains Hg, Pb, As, Cd etc.
    # Metadata
    data_confidence:  str = ''        # notes on data reliability


def classify_hazard(composite_score: float,
                     n_hits: int,
                     n_assays: int) -> Tuple[str, str]:
    """
    Assign hazard flag and confidence level.

    Hazard flags:
      HIGH     — composite >= 60 or >= 4 assay hits
      MEDIUM   — composite >= 30 or >= 2 assay hits
      LOW      — composite >= 10 or 1 assay hit
      NEGATIVE — composite < 10 and 0 assay hits

    Confidence is lower when few assays were tested.
    """
    if composite_score >= 60 or n_hits >= 4:
        flag = 'HIGH'
    elif composite_score >= 30 or n_hits >= 2:
        flag = 'MEDIUM'
    elif composite_score >= 10 or n_hits >= 1:
        flag = 'LOW'
    else:
        flag = 'NEGATIVE'

    # Confidence based on assay coverage
    if n_assays >= 12:
        conf = 'HIGH'
    elif n_assays >= 6:
        conf = 'MODERATE'
    else:
        conf = 'LOW'

    return flag, conf


def bbb_penetration_concern(phys: Dict) -> bool:
    """
    Simplified BBB penetration estimate (Lipinski-inspired).
    Concern: LogP >= 1.5, MW <= 500, TPSA <= 90 Da^2
    """
    return (phys.get('LogP', 0) >= 1.5 and
            phys.get('MW', 999) <= 500 and
            phys.get('TPSA', 999) <= 90)


# ── Build full profiles ────────────────────────────────────────────────────────
profiles: List[NeurotoxicityProfile] = []

for i, cid in enumerate(feature_ids):
    rec   = next(r for r in valid if r.input_id == cid)
    phys  = compute_physicochemical(rec.mol)
    asc   = assay_scores[cid]

    ml_s  = float(ensemble_proba[i]) * 100
    as_s  = asc['assay_score']

    # Composite: 60% ML probability + 40% assay evidence
    composite = 0.60 * ml_s + 0.40 * as_s

    flag, conf = classify_hazard(composite, asc['n_hits'], asc['n_assays_tested'])

    # Data confidence note
    notes = []
    if asc['n_hits'] == 0:
        notes.append('No assay hits detected')
    if asc['n_assays_tested'] < 8:
        notes.append('Limited assay coverage')
    if phys.get('HeavyMetals', 0) > 0:
        notes.append('Contains heavy metal — fingerprint-based ML unreliable')

    prof = NeurotoxicityProfile(
        chemical_id     = cid,
        name            = names.get(cid, cid),
        smiles          = rec.smiles,
        inchikey        = rec.inchikey,
        ml_score        = round(ml_s, 2),
        assay_score     = round(as_s, 2),
        composite_score = round(composite, 2),
        hazard_flag     = flag,
        confidence      = conf,
        n_assay_hits    = asc['n_hits'],
        mechanisms      = asc['mechanisms'],
        aops_triggered  = asc['aops_triggered'],
        key_assay_hits  = asc['hit_assays'][:5],
        bbb_concern     = bbb_penetration_concern(phys),
        heavy_metal     = phys.get('HeavyMetals', 0) > 0,
        data_confidence = '; '.join(notes) if notes else 'Adequate'
    )
    profiles.append(prof)


# ── Summary table ─────────────────────────────────────────────────────────────
profile_df = pd.DataFrame([asdict(p) for p in profiles])
profile_df = profile_df.sort_values('composite_score', ascending=False)

FLAG_ICONS = {'HIGH':'[!!!]','MEDIUM':'[!!]','LOW':'[!]','NEGATIVE':'[ ]'}
print('Neurotoxicity Profiler — Results')
print('='*75)
print(f'{"Chemical":18s} {"ML":>6s} {"Assay":>6s} {"Score":>6s} {"Flag":>9s} {"Conf":>8s}  Mechanisms')
print('-'*75)
for p in sorted(profiles, key=lambda x: -x.composite_score):
    icon = FLAG_ICONS.get(p.hazard_flag, '')
    mech = ', '.join(p.mechanisms[:2]) if p.mechanisms else 'none'
    print(f'{p.chemical_id:18s} {p.ml_score:6.1f} {p.assay_score:6.1f} {p.composite_score:6.1f} '
          f'{icon+p.hazard_flag:>12s} {p.confidence:>8s}  {mech}')

---
## Module 7: Visualization — Risk Dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

FLAG_COLORS = {'HIGH':'#c0392b','MEDIUM':'#e67e22','LOW':'#f1c40f','NEGATIVE':'#27ae60'}

fig = plt.figure(figsize=(16, 12))
fig.suptitle('Neurotoxicity Profiler — Risk Dashboard', fontsize=15, fontweight='bold', y=0.98)

ax_bar   = fig.add_subplot(2, 3, 1)
ax_score = fig.add_subplot(2, 3, 2)
ax_assay = fig.add_subplot(2, 3, 3)
ax_phys  = fig.add_subplot(2, 1, 2)

# 1. Composite score bar chart ─────────────────────────────────────────────────
sorted_profs = sorted(profiles, key=lambda x: x.composite_score, reverse=True)
cids   = [p.chemical_id for p in sorted_profs]
scores = [p.composite_score for p in sorted_profs]
flags  = [p.hazard_flag for p in sorted_profs]
colors = [FLAG_COLORS[f] for f in flags]

bars = ax_bar.barh(cids[::-1], scores[::-1], color=colors[::-1], edgecolor='white', height=0.7)
ax_bar.axvline(60, color='#c0392b', ls='--', lw=1.2, alpha=0.6, label='HIGH threshold')
ax_bar.axvline(30, color='#e67e22', ls='--', lw=1.2, alpha=0.6, label='MEDIUM threshold')
ax_bar.set_xlabel('Composite Score (0-100)')
ax_bar.set_title('Composite Neurotoxicity Score')
ax_bar.legend(fontsize=8)
ax_bar.set_xlim(0, 105)
for bar, score in zip(bars, scores[::-1]):
    ax_bar.text(score + 1, bar.get_y() + bar.get_height()/2,
                f'{score:.1f}', va='center', fontsize=8)

# 2. ML vs Assay scatter ───────────────────────────────────────────────────────
for p in profiles:
    ax_score.scatter(p.ml_score, p.assay_score,
                      c=FLAG_COLORS[p.hazard_flag], s=80, edgecolor='gray', linewidth=0.5, zorder=5)
    ax_score.annotate(p.chemical_id, (p.ml_score, p.assay_score),
                       fontsize=7, xytext=(3, 3), textcoords='offset points')
ax_score.axvline(50, color='gray', ls=':', lw=1)
ax_score.axhline(30, color='gray', ls=':', lw=1)
ax_score.set_xlabel('ML Score (0-100)')
ax_score.set_ylabel('Assay Score (0-100)')
ax_score.set_title('ML vs. Assay Evidence')
ax_score.set_xlim(-5, 110); ax_score.set_ylim(-5, 75)
patches = [mpatches.Patch(color=c, label=f) for f, c in FLAG_COLORS.items()]
ax_score.legend(handles=patches, fontsize=7)

# 3. Assay hit heatmap ─────────────────────────────────────────────────────────
short_assays = [a.replace('Tox21_','T21_').replace('NVS_','') for a in assay_names]
sorted_ids   = [p.chemical_id for p in sorted_profs]
heat_data    = assay_df.loc[sorted_ids][assay_names]
sns.heatmap(heat_data, ax=ax_assay, cmap='RdYlGn_r', cbar=False,
            xticklabels=short_assays, yticklabels=sorted_ids,
            linewidths=0.4, linecolor='white', vmin=0, vmax=1)
ax_assay.set_title('Assay Hit Heatmap')
ax_assay.tick_params(axis='x', labelsize=6, rotation=45)
ax_assay.tick_params(axis='y', labelsize=7)

# 4. Physicochemical radar / parallel plot ─────────────────────────────────────
display_cols = ['LogP','MW','TPSA','HBD','AromaticRings','Halogens']
phys_plot    = physchem_df[display_cols].copy()
# Normalise for display
norms_display = {'LogP':8,'MW':700,'TPSA':150,'HBD':6,'AromaticRings':5,'Halogens':10}
for col in display_cols:
    phys_plot[col] = phys_plot[col] / norms_display[col]

for cid in sorted_ids:
    flag = next(p.hazard_flag for p in profiles if p.chemical_id == cid)
    ax_phys.plot(display_cols, phys_plot.loc[cid].values,
                  color=FLAG_COLORS[flag], alpha=0.6, lw=1.5, marker='o', markersize=4)

ax_phys.set_ylabel('Normalised value')
ax_phys.set_title('Physicochemical Profile (normalised) — by Hazard Flag')
ax_phys.set_ylim(0, 1.2)
ax_phys.grid(alpha=0.3)
ax_phys.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('/home/claude/neuro_profiler_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Dashboard saved.')

---
## Module 8: Mechanistic AOP Report Generator

In [ ]:
from datetime import datetime


def generate_report(profile: NeurotoxicityProfile,
                     format: str = 'text') -> str:
    """
    Generate a structured neurotoxicity assessment report.
    format: 'text' | 'markdown' | 'json'
    """
    ts = datetime.now().strftime('%Y-%m-%d %H:%M')

    aop_text = ''
    for aop in profile.aops_triggered:
        aop_text += f"\n      {aop.get('aop_id','N/A')}: {aop.get('title','')}"
        aop_text += f"\n        MIE: {aop.get('mie','')}"
        aop_text += f"\n        AO:  {aop.get('ao','')}"

    bbb_str = 'YES — physicochemical properties suggest CNS penetration concern' if profile.bbb_concern else 'Low concern'
    hm_str  = 'YES — heavy metal detected; QSAR reliability limited' if profile.heavy_metal else 'Not detected'

    text = f"""\
╔══════════════════════════════════════════════════════════════╗
║       NEUROTOXICITY PROFILER — ASSESSMENT REPORT            ║
╚══════════════════════════════════════════════════════════════╝
Generated: {ts}

CHEMICAL IDENTIFICATION
  ID:        {profile.chemical_id}
  Name:      {profile.name}
  SMILES:    {(profile.smiles or 'N/A')[:60]}
  InChIKey:  {profile.inchikey or 'N/A'}

HAZARD CLASSIFICATION
  Hazard Flag:       {profile.hazard_flag}   ({'CONCERN' if profile.hazard_flag != 'NEGATIVE' else 'NOT FLAGGED'})
  Confidence Level:  {profile.confidence}

RISK SCORES (0–100)
  ML Ensemble Score: {profile.ml_score:>6.1f}  (Random Forest + XGBoost, molecular fingerprints)
  Assay Score:       {profile.assay_score:>6.1f}  (Weighted ToxCast/Tox21 activity, {len(NEURO_ASSAY_PANEL)} assay panel)
  Composite Score:   {profile.composite_score:>6.1f}  (60% ML + 40% assay evidence)

IN VITRO ASSAY EVIDENCE
  Assays tested:  {profile.n_assay_hits} hits / {len(NEURO_ASSAY_PANEL)} panel assays
  Active assays:  {', '.join(profile.key_assay_hits) or 'None'}

MECHANISMS OF CONCERN
  {'; '.join(profile.mechanisms) if profile.mechanisms else 'No mechanistic alerts detected'}

ADVERSE OUTCOME PATHWAYS (AOP-Wiki)
  {aop_text.strip() if aop_text else 'No AOPs triggered'}

PHYSICOCHEMICAL FLAGS
  BBB penetration concern: {bbb_str}
  Heavy metal concern:     {hm_str}

DATA RELIABILITY
  {profile.data_confidence}

INTERPRETATION NOTES
  - Composite score >= 60: HIGH concern; recommend in vivo follow-up
  - Composite score 30-59: MEDIUM concern; additional assays warranted
  - Composite score 10-29: LOW concern; monitor in context of exposure
  - Composite score  < 10: NEGATIVE; insufficient evidence for concern
  - Results do not substitute for full regulatory neurotoxicity testing
  - Model trained on limited demo data; production requires 500+ chemicals
"""
    return text


# Print report for highest-scoring chemical
top_chemical = max(profiles, key=lambda p: p.composite_score)
print(generate_report(top_chemical))

# Export all profiles to JSON
with open('/home/claude/neuro_profiles.json', 'w') as f:
    json.dump([asdict(p) for p in profiles], f, indent=2)
print('\nAll profiles saved to neuro_profiles.json')

---
## Module 9: Export, API Wrapper & Production Notes

In [ ]:
# ── NeurotoxProfiler class — production-ready wrapper ─────────────────────────

class NeurotoxProfiler:
    """
    Production-ready neurotoxicity profiler.

    Usage:
        profiler = NeurotoxProfiler()
        profiler.fit(training_smiles, training_labels, training_assay_df)
        profile  = profiler.predict(smiles='CCc1ccc(O)cc1')
        print(profile.hazard_flag, profile.composite_score)
    """

    def __init__(self,
                 fp_type:       str   = 'combined',
                 rf_estimators: int   = 300,
                 ensemble_weight_ml:    float = 0.60,
                 ensemble_weight_assay: float = 0.40):
        self.fp_type = fp_type
        self.w_ml    = ensemble_weight_ml
        self.w_assay = ensemble_weight_assay
        self.rf      = RandomForestClassifier(n_estimators=rf_estimators,
                                               class_weight='balanced',
                                               random_state=42, n_jobs=-1)
        try:
            import xgboost as xgb
            self.xgb = xgb.XGBClassifier(n_estimators=200, max_depth=4,
                                          learning_rate=0.05, random_state=42,
                                          verbosity=0)
        except ImportError:
            from sklearn.ensemble import GradientBoostingClassifier
            self.xgb = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                                   random_state=42)
        self.fitted = False

    def fit(self, smiles_list: List[str], labels: List[int],
             assay_df: Optional[pd.DataFrame] = None) -> 'NeurotoxProfiler':
        """Train both models on labelled chemical data."""
        records = [parse_smiles(s, f'chem_{i}') for i, s in enumerate(smiles_list)]
        X_fp    = np.array([featurize(r, self.fp_type) for r in records
                             if r.valid and featurize(r, self.fp_type) is not None])
        y_fit   = np.array(labels)

        if assay_df is not None:
            X_fit = np.hstack([X_fp, assay_df.values])
        else:
            X_fit = X_fp

        self.rf.fit(X_fit, y_fit)
        self.xgb.fit(X_fit, y_fit)
        self.n_features = X_fit.shape[1]
        self.fitted = True
        print(f'Fitted on {len(y_fit)} chemicals, {X_fit.shape[1]} features')
        return self

    def predict(self, smiles: str = '',
                inchi_str: str = '',
                name: str = '',
                assay_hits: Optional[Dict] = None) -> NeurotoxicityProfile:
        """Score a single chemical and return NeurotoxicityProfile."""
        assert self.fitted, 'Call .fit() first.'

        rec = parse_smiles(smiles) if smiles else parse_inchi(inchi_str)
        if not rec.valid:
            return NeurotoxicityProfile(
                chemical_id='unknown', name=name, smiles=smiles,
                inchikey=None, ml_score=0, assay_score=0,
                composite_score=0, hazard_flag='INSUFFICIENT_DATA',
                confidence='LOW', n_assay_hits=0)

        fp   = featurize(rec, self.fp_type)
        if assay_hits:
            assay_vec = np.array([float(assay_hits.get(a, 0)) for a in assay_names])
            X_pred = np.hstack([fp, assay_vec]).reshape(1, -1)
        else:
            X_pred = fp.reshape(1, -1)

        # Pad/truncate if needed
        if X_pred.shape[1] < self.n_features:
            X_pred = np.pad(X_pred, ((0,0),(0, self.n_features - X_pred.shape[1])))
        elif X_pred.shape[1] > self.n_features:
            X_pred = X_pred[:, :self.n_features]

        rf_p  = self.rf.predict_proba(X_pred)[0, 1] * 100
        xgb_p = self.xgb.predict_proba(X_pred)[0, 1] * 100
        ml_s  = (rf_p + xgb_p) / 2

        asc   = compute_assay_score(assay_hits or {}, NEURO_ASSAY_PANEL)
        comp  = self.w_ml * ml_s + self.w_assay * asc['assay_score']
        flag, conf = classify_hazard(comp, asc['n_hits'], asc['n_assays_tested'])
        phys  = compute_physicochemical(rec.mol)

        return NeurotoxicityProfile(
            chemical_id     = rec.inchikey or 'unknown',
            name            = name,
            smiles          = rec.smiles,
            inchikey        = rec.inchikey,
            ml_score        = round(ml_s, 2),
            assay_score     = round(asc['assay_score'], 2),
            composite_score = round(comp, 2),
            hazard_flag     = flag,
            confidence      = conf,
            n_assay_hits    = asc['n_hits'],
            mechanisms      = asc['mechanisms'],
            aops_triggered  = asc['aops_triggered'],
            key_assay_hits  = asc['hit_assays'],
            bbb_concern     = bbb_penetration_concern(phys),
            heavy_metal     = phys.get('HeavyMetals', 0) > 0
        )


# Quick demo of the class-based API
demo_profiler = NeurotoxProfiler()
demo_profiler.fit(
    smiles_list = [rec.smiles for rec in valid],
    labels      = [labels.get(rec.input_id, 0) for rec in valid],
    assay_df    = assay_df
)

# Score a new compound — atrazine (herbicide, developmental neurotox concern)
atrazine_hits = {'NVS_NR_hTRa_Antagonist':1, 'TOX21_NFKB_BLA_Agonist':1,
                  'Tox21_ARE_BLA_Agonist':0, 'NVS_ENZ_hAChE':0}
new_profile = demo_profiler.predict(
    smiles     = 'CCNc1nc(Cl)nc(NC(C)C)n1',
    name       = 'Atrazine',
    assay_hits = atrazine_hits
)
print('\nDemo prediction — Atrazine:')
print(generate_report(new_profile))

---
## Production Roadmap & Scaling Guide

### Data Sources for Production Model
| Dataset | n chemicals | Access |
|---|---|---|
| ToxCast Phase I/II | ~1,800 | Free — EPA CompTox API |
| Tox21 Challenge | ~8,000 | Free — NCATS |
| ToxRefDB (in vivo) | ~1,000 | Free — EPA |
| ChemIDplus neurotox | ~500 | Free — NLM |
| ECOTOX (vertebrate) | ~5,000+ | Free — EPA |

### Model Improvements
- **Graph Neural Networks (GNN):** Replace fingerprints with `torch_geometric` GCN/MPNN — learns directly on molecular graphs; state-of-the-art for QSAR
- **Multi-task learning:** Predict all 17 assay endpoints simultaneously rather than binary active/inactive
- **Uncertainty quantification:** Add conformal prediction intervals — critical for regulatory use
- **Applicability domain:** Flag chemicals outside training set distribution (Tanimoto similarity cutoff)

### Tool Integrations
```python
# DeepChem — GNN models, ADMET prediction
import deepchem as dc
featurizer = dc.feat.MolGraphConvFeaturizer()

# ChemBERTa — transformer on SMILES strings
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')

# OPERA — regulatory QSAR suite (EPA ORD)
# https://github.com/kmansouri/OPERA
```

### Deployment
```bash
# FastAPI REST endpoint
# POST /predict  {"smiles": "CCc1ccc(O)cc1", "assay_hits": {...}}
# → NeurotoxicityProfile JSON

# Streamlit dashboard (local / HuggingFace Spaces)
streamlit run neuro_profiler_app.py

# Docker deployment
docker build -t neuro-profiler .
docker run -p 8000:8000 neuro-profiler
```

### Regulatory Context
- **EPA TSCA:** Neurotoxicity testing guidance under TSCA Section 4
- **OECD TG 424/426:** In vivo neurotoxicity guidelines — use this tool to prioritise
- **ICH S7A:** Nonclinical safety pharmacology for CNS — relevant for drug development
- **EFSA NAM guidance:** New Approach Methods replacing animal tests
- **AOP-Wiki:** https://aopwiki.org — source for AOP mechanistic annotations

### Essential reading
- Kavlock & Dix (2010) — ToxCast program overview
- Tice et al. (2013) — Improving the human hazard characterization of chemicals (Tox21)
- Leist et al. (2014) — Consensus report on the future of animal-free systemic toxicity testing
- Bal-Price et al. (2018) — Adverse Outcome Pathways for developmental neurotoxicity
- Yang et al. (2019) — Analyzing learned molecular representations (ChemProp / MPNN)